# Chip-creation example: polar AOI

Create WAC, NAC or static-only chips from geographic bounds and finished labels. No reference chip is required. For ordinary LTM imagery, use [the regular example](chip_example.ipynb).

Select **lfm_kernel**, edit the inputs, and run cells in order. Dynamic chips keep the native source grid; static-only chips use a 100 m center-selected grid. Pole caps retain the enclosing rectangle's corners without masking.

Like the regular example: no dataset splits, one chip worker, allocated CPUs for indexing, and overwrite on rerun. The established north-seam query remains enabled; other validation cases are opt-in. The appendix preserves the full edge-case checklist and regression tools. Untested combinations still need real-data validation.

## Imports and repository discovery

Run from the repository's top-level `notebooks/` directory. Repository discovery normalizes JupyterHub's `/panfs` path to `/explore` before importing LFM, as in the regular example.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys

repo_root = Path(str(Path.cwd().parent).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"))
if not (repo_root / "lfm/data_processing").is_dir():
    raise FileNotFoundError("Run from the repository's top-level notebooks/ directory.")
sys.path.insert(0, str(repo_root))

from lfm.data_processing.chip import (
    AcquisitionGroupConfig, ChipConfig, GeographicAOI, LabelInput,
    NoSplitConfig, OutputModalityConfig, SourceSelector,
    chip_grid_family, default_chip_zoom, chip_request_from_aoi, create_chips,
)
from lfm.data_processing.tiling import (
    TileConfig, TileSourceConfig, VectorIndexBuildConfig,
    ensure_vector_index, resolve_notebook_source_index, discover_raster_paths,
)
from lfm.data_processing.tiling.product_ids import lunar_product_id_from_raster_path
from lfm.all_models.all_tasks.tiling_utils import make_static_source
from lfm.data_processing.chip.chip_notebook_utils import (
    latest_crater_label_path, notebook_index_workers, read_source_grid,
    plot_chip_result, chip_batch_fingerprints,
)

print(f"Repository root: {repo_root}")


## 1. Choose imagery, AOI and labels

Variables below follow code order. Update source, hemisphere, bounds and labels together; changing hemisphere does not select different labels.

- `PROJECT_DATA_DIR`: shared data root.
- `HEMISPHERE`: north or south.
- `MODALITY`: `wac`, `nac`, or `static` for static-only output.
- `INCLUDE_STATIC`: append 63 static bands; initially disabled.
- `STATIC_DATA_DIR`: inspected only when static is enabled.
- `OUTPUT_BASE_DIR`: timestamped runs beneath `notebooks/outputs/`.
- `SOURCE_RASTER`: original raster or VRT. `None` discovers the hemisphere's WAC mosaic; NAC requires an explicit path.
- `WAC_LAYOUT`: `single` for monochrome polar WAC, or `vis_uv` for five VIS plus two UV bands.
- `DYNAMIC_GLOB`: optional companion-file pattern, otherwise derived from the source/layout.
- `POLAR_WAC_DIR`: collection searched for the hemisphere-specific VRT.
- `AOI_NWSE`: north, west, south, east in IAU:30100. North defaults cover supplied craters 13–20; south needs its own labels.
- `LABEL_PATH`: finished GPKG or integer TIFF; `None` discovers the latest crater export without matching its study area.
- `LABEL_LAYER`: GeoPackage layer; ignored for TIFF.
- `LABEL_KIND`: `auto` for crater GPKGs or semantic TIFFs; `raster_instance` for instance TIFFs.
- `SAMPLE_ID`: output identity.
- `ZOOM_OVERRIDE`: optional acquisition zoom override, not output resolution.
- `MAX_OUTPUT_PIXELS`: size guard before acquisition, especially important for NAC; the main request is checked before indexing too.

Raster labels must cover the whole realized rectangle. Supplied annotations do not establish completeness at either pole or the antimeridian.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
HEMISPHERE = "north"  # "north" or "south".
MODALITY = "wac"  # "wac", "nac", or "static".
INCLUDE_STATIC = False  # Start with imagery only; verify static coverage separately.
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/chip_creation_polar"
SOURCE_RASTER = None  # Set an explicit Path for your source, especially NAC.
WAC_LAYOUT = "single"  # Polar mosaic is not the seven-band VIS/UV product.
DYNAMIC_GLOB = None  # Optional glob selecting this product's companion files.
POLAR_WAC_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/Static_final/LROC/WAC/wac_glob_morf_mos"
AOI_NWSE = (86.1, -48.7, 83.95, -19.5) if HEMISPHERE == "north" else (-85.9, -0.1, -86.1, 0.1)
LABEL_PATH = PROJECT_DATA_DIR / "example_notebook_inputs" / "WAC_GLOBAL_P900N0000_100M.eqc.iau2.LPS_N_label_craters.gpkg"  # Set your HPC copy's path if different.
LABEL_LAYER = "craters"
LABEL_KIND = "auto"  # For instance TIFFs use "raster_instance".
SAMPLE_ID = f"{MODALITY}_{HEMISPHERE}_aoi_1"  # Unique per study area.
ZOOM_OVERRIDE = None
MAX_OUTPUT_PIXELS = 4_000_000  # Bounds per-band memory; static adds 63 bands.

### Optional polar validation queries

The north WAC seam example stays enabled; disable it for one chip only. Other checks are opt-in. The appendix explains coverage and expected outcomes.

- `RUN_SEAM_EXAMPLE` / `SEAM_AOI_NWSE`: add the labeled north query crossing 82°; supply appropriate bounds and labels for south.
- `RUN_POLE_EXAMPLE` / `POLE_AOI_NWSE`: add a full-longitude cap reaching ±90°; retain rectangle corners.
- `EXTRA_REAL_CASES`: antimeridian, combined seam/antimeridian, and exact-boundary presets.
- `CASE_LABEL_PATHS`: per-case finished labels; otherwise use the main path.
- `RUN_PARALLEL_REPLAY`: two extra isolated real-data runs.
- `REPLAY_WORKERS`: workers for the optional parallel comparison.
- `RUN_REGRESSIONS`: synthetic tests in a fresh interpreter; skips mean incomplete validation.

In [ ]:
RUN_SEAM_EXAMPLE = HEMISPHERE == "north" and MODALITY == "wac"  # Add the labeled north seam query.
SEAM_AOI_NWSE = (86.1, -48.7, 81.95, -19.5)  # Includes IDs 13–20 and crosses 82°N.
RUN_POLE_EXAMPLE = False  # Enable after confirming source and label coverage near the pole.
POLE_AOI_NWSE = (90, -180, 89.5, 180) if HEMISPHERE == 'north' else (-89.5, -180, -90, 180)
EXTRA_REAL_CASES = ()  # e.g. ("antimeridian", "seam_antimeridian", "boundary_polar", "boundary_ltm")
CASE_LABEL_PATHS = {}  # e.g. {"antimeridian": Path("/path/to/finished_labels.gpkg")}
RUN_PARALLEL_REPLAY = False  # Two extra real-data runs; use at least two requests.
REPLAY_WORKERS = 2
RUN_REGRESSIONS = False  # Enable on the GDAL-enabled HPC kernel.


## 2. Resolve paths and prepare indexes

Derived values follow code order:

- `requested_aoi` / `family` / `expected_family`: bounds, center/sole grid family and selected hemisphere.
- `static_only` / `source_grid` / `selectors` / `GROUP_NAME`: native-grid and imagery selection setup.
- `PRODUCT_ID`: exact source product selector, independent of labels.
- `label_input`: explicit labels and clipping contract.
- `request`: built early to check output size before indexing.
- `ZOOM_LEVEL` / `LTM_ZOOM` / `POLAR_ZOOM`: acquisition zooms; defaults WAC/static 5/4, NAC 11/10.
- `OUTPUT_ROOT`: unique run destination.
- `requests`: main request plus optional queries added in step 3.

The output rectangle is authoritative. Known uncovered bands retain -32768 with warnings; corrupt inputs still fail. The size check intentionally precedes indexing.

In [ ]:
if HEMISPHERE not in ("north", "south") or MODALITY not in ("wac", "nac", "static"):
    raise ValueError("Choose a supported hemisphere and modality.")
if WAC_LAYOUT not in ("single", "vis_uv"):
    raise ValueError("WAC_LAYOUT must be 'single' or 'vis_uv'.")
requested_aoi = GeographicAOI(*AOI_NWSE)
family = chip_grid_family(requested_aoi)
expected_family = "lps_n" if HEMISPHERE == "north" else "lps_s"
if not (requested_aoi.upper_left_latitude > 82 if HEMISPHERE == 'north' else requested_aoi.lower_right_latitude < -82):
    raise ValueError("AOI must reach the selected polar hemisphere; crossing its 82° seam is allowed.")
static_only = MODALITY == "static"
source_grid = None
selectors = ()
GROUP_NAME = "polar"
if not static_only:
    if SOURCE_RASTER is None:
        if MODALITY != "wac" or WAC_LAYOUT != "single":
            raise ValueError("Set SOURCE_RASTER to your original imagery raster.")
        matches = sorted(POLAR_WAC_DIR.glob(f"*{expected_family.upper()}.vrt"))
        if len(matches) != 1:
            raise ValueError(f"Expected one polar VRT; found {len(matches)}. Set SOURCE_RASTER explicitly.")
        SOURCE_RASTER = matches[0]
    SOURCE_RASTER = Path(SOURCE_RASTER)
    source_grid = read_source_grid(
        SOURCE_RASTER, expected_band_count=5 if MODALITY == "wac" and WAC_LAYOUT == "vis_uv" else 1)
    PRODUCT_ID = lunar_product_id_from_raster_path(SOURCE_RASTER)
    selectors = (SourceSelector(GROUP_NAME, MODALITY, PRODUCT_ID),)
LABEL_PATH = latest_crater_label_path() if LABEL_PATH is None else Path(LABEL_PATH)
if not LABEL_PATH.is_file():
    raise FileNotFoundError(LABEL_PATH)
label_input = LabelInput(LABEL_PATH, kind=LABEL_KIND, relation="clip_to_target",
                         layer=LABEL_LAYER if LABEL_PATH.suffix.lower() == ".gpkg" else None)
request = chip_request_from_aoi(
    sample_id=SAMPLE_ID, geographic_aoi=requested_aoi,
    source_grid=source_grid, static_only=static_only,
    split_group_key=SAMPLE_ID if static_only else PRODUCT_ID,
    source_selectors=selectors, label_input=label_input,
)
if request.target_grid.width * request.target_grid.height > MAX_OUTPUT_PIXELS:
    raise ValueError("Output exceeds MAX_OUTPUT_PIXELS; reduce AOI or explicitly raise the limit.")
ZOOM_LEVEL = default_chip_zoom(request.geographic_aoi, MODALITY) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
LTM_ZOOM = (11 if MODALITY == 'nac' else 5) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
POLAR_ZOOM = (10 if MODALITY == 'nac' else 4) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
OUTPUT_ROOT = OUTPUT_BASE_DIR / datetime.now().strftime("%Y%m%dT%H%M%S_%f")
print(f"Labels: {LABEL_PATH}")
print(f"Center/sole family: {family.value} | acquisition zooms: LTM={LTM_ZOOM}, polar={POLAR_ZOOM}")
print(f"Output: {request.target_grid.width} x {request.target_grid.height} | affine: {request.target_grid.transform}")
print(f"Destination: {OUTPUT_ROOT}")
requests = [request]


### Configure indexes and output bands

- `sources` / `modalities`: enabled inputs and output band order.
- `pattern` / `index_config` / `index`: product-scoped dynamic discovery and this run's index.
- `single`: selects one NAC or monochrome WAC band; otherwise VIS 0–4 then UV 5–6.
- `resolution`: shared-static index or custom-cache selection.
- `chip_config`: pipeline configuration without train/val/test splits.

Dynamic indexing selects only the chosen product, avoiding duplicate VRT/backing-TIFF inputs. Shared static indexes remain read-only. Index workers use the Slurm allocation or available CPUs, as in the regular example.

In [ ]:
sources, modalities = [], []
if not static_only:
    pattern = DYNAMIC_GLOB or (
        f"{PRODUCT_ID}.prj.*.mos.tif" if MODALITY == "wac" and WAC_LAYOUT == "vis_uv"
        else SOURCE_RASTER.name)
    index_config = VectorIndexBuildConfig(
        data_dir=SOURCE_RASTER.parent, index_path=OUTPUT_ROOT / "indexes/dynamic.gpkg",
        image_glob=pattern, worker_count=notebook_index_workers())
    if SOURCE_RASTER.resolve() not in {p.resolve() for p in discover_raster_paths(index_config)}:
        raise ValueError('DYNAMIC_GLOB must include SOURCE_RASTER.')
    index = ensure_vector_index(index_config, stdout=sys.stdout).index_path
    sources.append(TileSourceConfig(
        name=MODALITY, data_dir=SOURCE_RASTER.parent, index_path=index,
        selection_mode="product_id", preserve_source_nodata=True, required=True))
    single = MODALITY == "nac" or WAC_LAYOUT == "single"
    modalities.append(OutputModalityConfig(
        GROUP_NAME, MODALITY, MODALITY,
        band_indices=(1,) if single else None,
        output_band_names=(MODALITY,) if single else None))
if static_only or INCLUDE_STATIC:
    resolution = resolve_notebook_source_index(
        source_name="static", data_dir=STATIC_DATA_DIR, cache_dir=OUTPUT_BASE_DIR / "indexes")
    index = ensure_vector_index(VectorIndexBuildConfig(
        data_dir=resolution.data_dir, index_path=resolution.index_path,
        rebuild_invalid_index=resolution.rebuild_invalid_index,
        worker_count=notebook_index_workers()), stdout=sys.stdout).index_path
    sources.append(make_static_source(data_dir=STATIC_DATA_DIR, index_path=index))
    modalities.append(OutputModalityConfig(GROUP_NAME, "static", "static"))
chip_config = ChipConfig(
    output_root=OUTPUT_ROOT, label_source=LABEL_PATH,
    acquisition_groups=(AcquisitionGroupConfig(
        GROUP_NAME, TileConfig(OUTPUT_ROOT / ".tiling-template", ZOOM_LEVEL, tuple(sources)),
        ltm_zoom_level=LTM_ZOOM, polar_zoom_level=POLAR_ZOOM),),
    output_modalities=tuple(modalities), split_config=NoSplitConfig(),
)


## 3. Build and inspect the request

The main `request` was built in step 2 for its early size check. `request_summary` shows the requested AOI, realized query, native CRS/affine, dimensions and label path. Inspect these before creation.

The optional cells below append distinct requests on the same source grid. Seam acquisition spans grid families, but labels are prepared once per output grid.

In [ ]:
request_summary = {
    "sample_id": request.sample_id,
    "requested_aoi_iau30100": request.requested_aoi.to_dict(),
    "realized_query_iau30100": request.geographic_aoi.to_dict(),
    "output_grid": request.target_grid.to_dict(),
    "label_source": str(LABEL_PATH),
}
request_summary


### Additional query: labeled north seam

This reuses the source grid, labels and product selector, with a distinct sample ID. The default AOI includes craters 13–20 from `(2).gpkg` and extends below 82°N (about 2.2 million pixels at 100 m). No supplied outlines cross the seam itself: inspect imagery continuity there and label alignment farther north. The output rectangle may extend beyond the geographic AOI. Both results share one manifest and are plotted below.


In [ ]:
requests = [request]  # Reset so rerunning this cell never duplicates the query.
if RUN_SEAM_EXAMPLE:
    seam_aoi = GeographicAOI(*SEAM_AOI_NWSE)
    boundary = 82 if HEMISPHERE == "north" else -82
    if not seam_aoi.lower_right_latitude < boundary < seam_aoi.upper_left_latitude:
        raise ValueError("SEAM_AOI_NWSE must cross the selected hemisphere's 82° seam.")
    seam_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_seam", geographic_aoi=seam_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = seam_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Seam output exceeds MAX_OUTPUT_PIXELS; reduce SEAM_AOI_NWSE.")
    requests.append(seam_request)
    print(f"Seam query: {SEAM_AOI_NWSE} | {grid.width} x {grid.height} pixels")


### Optional query: pole-containing rectangle

Enable `RUN_POLE_EXAMPLE` only with imagery and finished labels covering the pole. A full-longitude cap becomes an enclosing native-grid rectangle: corner imagery and labels outside the cap are retained, not masked. Acquisition covers that entire rectangle. The supplied example annotations are farther from the pole, so an all-background result here is not evidence of a correctly annotated negative sample. Large rectangles extending below the polar coverage boundary are rejected.


In [ ]:
requests = [item for item in requests if item.sample_id != f"{SAMPLE_ID}_pole"]
if RUN_POLE_EXAMPLE:
    pole_aoi = GeographicAOI(*POLE_AOI_NWSE)
    pole_latitude = pole_aoi.upper_left_latitude if HEMISPHERE == "north" else pole_aoi.lower_right_latitude
    if pole_latitude != (90 if HEMISPHERE == "north" else -90):
        raise ValueError("POLE_AOI_NWSE must reach the selected hemisphere's pole.")
    pole_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_pole", geographic_aoi=pole_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = pole_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Pole output exceeds MAX_OUTPUT_PIXELS; reduce the cap.")
    requests.append(pole_request)
    print(f"Pole query: {POLE_AOI_NWSE} | {grid.width} x {grid.height} pixels; corners retained")
    print("Acquisition envelope:", pole_request.geographic_aoi)


### Additional real-data geographic cases

These presets stay small for WAC; reduce them further for NAC. Exact-boundary inputs may acquire both families after the native rectangle is rounded outward—inspect the **realized** query parts, not just the input latitude. The combined seam/antimeridian case should not duplicate labels or introduce a blank stripe.

Presets use the selected source lattice and selectors. Unsupported source coverage is a meaningful failure, not an automatic reason to replace imagery or labels. Changing hemisphere does not change the label path for you.


In [ ]:
from lfm.data_processing.chip.chip_requests import geographic_query_parts

north_presets = {
    "antimeridian": (86.01, 179.98, 86.0, -179.98),
    "seam_antimeridian": (82.01, 179.98, 81.99, -179.98),
    "boundary_polar": (82.01, 10.0, 82.0, 10.02),
    "boundary_ltm": (82.0, 10.0, 81.99, 10.02),
}
presets = {name: bounds if HEMISPHERE == "north" else (-bounds[2], bounds[1], -bounds[0], bounds[3])
           for name, bounds in north_presets.items()}
unknown = set(EXTRA_REAL_CASES) - presets.keys()
if unknown:
    raise ValueError(f"Unknown extra cases: {sorted(unknown)}")
extra_ids = {f"{SAMPLE_ID}_{name}" for name in presets}
requests = [item for item in requests if item.sample_id not in extra_ids]  # Rerun-safe.
for name in dict.fromkeys(EXTRA_REAL_CASES):
    path = Path(CASE_LABEL_PATHS.get(name, LABEL_PATH))
    case_label = LabelInput(path, kind=LABEL_KIND, relation="clip_to_target",
                            layer=LABEL_LAYER if path.suffix.lower() == ".gpkg" else None)
    item = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_{name}", geographic_aoi=GeographicAOI(*presets[name]),
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key, source_selectors=selectors, label_input=case_label,
    )
    grid = item.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError(f"{name} exceeds MAX_OUTPUT_PIXELS; reduce its preset.")
    requests.append(item)
    print(name, "| labels:", path, "| pixels:", (grid.width, grid.height))
    print("Realized query parts:", geographic_query_parts(item.geographic_aoi))


## 4. Create the chip

`batch` contains the main request and enabled validation queries. Creation uses one worker, overwrite enabled, and a completion bar without worker-stage messages. Review each sample's status, manifest and diagnostics.

In [ ]:
batch = create_chips(
    requests, chip_config,
    max_workers=1, overwrite=True,
    progress=True, progress_mode="bar",
)
print(f"Processed {len(batch.results)} request(s) in {batch.elapsed_seconds:.1f}s")
print(f"Manifest: {batch.manifest_path}")
for result in batch.results:
    print(result.request.sample_id, result.status, result.message)
    for diagnostic in result.diagnostics:
        print(f"{diagnostic.stage}/{diagnostic.code}: {diagnostic.message}")


## 5. Inspect the chip and labels

The panels show imagery, labels and their overlay. Inspect alignment, clipped craters and seam/antimeridian stripes. For pole chips, inspect the center and unmasked corners. An empty mask outside the annotated area is not a verified negative sample.

`artifact` describes prepared labels and ID mappings; `figure_path` saves one figure per successful sample. Static-only uses the elevation band. Review per-band NoData in the manifest and diagnostics for failed samples.

In [ ]:
for result in batch.results:
    if result.prepared_label is not None:
        artifact = result.prepared_label
        print(f"Label source: {artifact.plan.source.path}")
        print(f"Preparation: {artifact.plan.method} | source → output IDs: {artifact.instance_id_map}")
    if result.status == "success":
        figure_path = OUTPUT_ROOT / f"{result.request.sample_id}_inspection.png"
        figure, axes = plot_chip_result(
            result, display_band_keyword="elv" if static_only else MODALITY,
            figure_path=figure_path,
        )
        print(f"Saved figure: {figure_path}")
    else:
        print(f"No pair published: {result.message}")


### Record real-data review

For every successful chip inspect native CRS/resolution, band order, imagery/label alignment, crater edges and boxes, and any seam/antimeridian stripe. For pole chips, inspect the center **and rectangle corners**: neither imagery nor labels should have a circular mask. Inspect static NoData per band, not just the displayed image band.

Set `REVIEW_NOTES` only after visual inspection. An empty mask outside the annotated area is not a verified negative sample. The saved record separates pipeline status from human review; no automatic visual pass is assigned.


In [ ]:
import json

REVIEW_NOTES = {}  # sample_id: "Reviewed: ..."; record missing labels/data as untested.
review = [
    {"sample_id": result.request.sample_id, "pipeline_status": result.status,
     "message": result.message, "visual_review": REVIEW_NOTES.get(result.request.sample_id, "NOT REVIEWED"),
     "chip": str(result.chip_path), "label": str(result.label_path),
     "diagnostics": [f"{d.stage}/{d.code}: {d.message}" for d in result.diagnostics]}
    for result in batch.results
]
review_path = OUTPUT_ROOT / "visual_review.json"
review_path.write_text(json.dumps(review, indent=2) + "\n")
print("Review record:", review_path)


## Outputs

Successful runs contain `chips/`, `labels/`, `diagnostics/`, `dataset_manifest.json`, inspection figures and the visual-review record. Review notes do not automatically certify samples. For dataset creation and splits, use [the full workflow](chip_full_workflow.ipynb).

In [ ]:
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_ROOT)}")


## Appendix A. Validation coverage

Run one hemisphere/source/label configuration at a time. Select extra real-data cases only after checking coverage; the presets are geometric examples, **not known annotated study areas**. Keep the output-size limit for NAC.

| Case | Real-data exercise | Automated synthetic check (Appendix C) |
|---|---|---|
| Antimeridian | `antimeridian` | Query split/deduplication and continuous labels |
| ±82° seam + antimeridian | `seam_antimeridian` | Four-part routing, family precedence and coverage fallback |
| Exact ±82° boundary | `boundary_polar`, `boundary_ltm` | Positive-area routing and pixel-center ownership |
| Pole inside chip | `RUN_POLE_EXAMPLE` | Both hemispheres, rectangle corners retained |
| Pole on edge/corner; rotated grid | Use a genuinely matching source/grid for real review | Native/rotated grids and pole-contact tests |
| Missing imagery/bands | Review warnings when static coverage is absent | Known-band NoData fill; empty acquisition/unknown bands fail |
| Bad/incomplete labels | Do not damage source files | Coverage/NoData/geometry failures, no published pair |
| Overlap, occlusion, clipping, tiny craters | Annotate examples, then crop through outlines | Compact IDs, clipped-outline boxes, overlap winner, omission warnings |
| Invalid geographic requests | Not sent to production acquisition | Partial-longitude poles, oversized caps and incompatible target grids reject |
| Serial/parallel and splits | Appendix B, separate output directories | Spawn equivalence, fixed-seed/group-atomic splits |
| Publication failures | Never induce failures in shared data | Rollback, overwrite/no-clobber and batch continuation |

Repeat applicable geographic cases for **north and south**, WAC/NAC, dynamic-only / dynamic+static / static-only, and GPKG / semantic TIFF / instance TIFF. TIFFs require full realized-grid coverage; `LABEL_KIND="raster_instance"` is required for instance TIFFs. These example runs are unsplit; synthetic tests check split behavior separately. Real NAC checks remain blocked until suitable imagery is available.


## Appendix B. Optional serial/parallel comparison

Runs the same prepared inputs through the public batch API twice, with isolated output/intermediate directories and unchanged read-only source indexes. Compare chip/label SHA-256 bytes and actual split assignments; exclude manifests' run-specific paths and timing. Requires at least two successful requests. A mismatch is a failure to investigate, not permission to overwrite either run. Synthetic split tests in Appendix C use multiple independent groups; this notebook's one-product batch cannot demonstrate meaningful train/val/test allocation.


In [ ]:
from dataclasses import replace

if RUN_PARALLEL_REPLAY:
    if len(requests) < 2 or REPLAY_WORKERS < 2:
        raise ValueError("Replay requires at least two requests and REPLAY_WORKERS >= 2.")
    replay_root = OUTPUT_ROOT / ("replay_" + datetime.now().strftime("%H%M%S_%f"))
    fingerprints = []
    for workers in (1, REPLAY_WORKERS):
        destination = replay_root / f"workers_{workers}"
        replay_config = replace(chip_config, output_root=destination,
                                intermediate_root=destination / ".intermediate")
        replay = create_chips(requests, replay_config, max_workers=workers)
        fingerprints.append(chip_batch_fingerprints(replay))  # Fails on unsuccessful/empty results.
    if fingerprints[0] != fingerprints[1]:
        raise AssertionError(f"Serial/parallel outputs differ; inspect {replay_root}")
    (replay_root / "comparison.json").write_text(json.dumps(
        {"status": "passed", "workers": [1, REPLAY_WORKERS], "fingerprints": fingerprints[0]}, indent=2) + "\n")
    print("Serial/parallel bytes and split assignments match:", replay_root)
else:
    print("Real-data serial/parallel comparison: NOT RUN")


## Appendix C. Synthetic edge cases and expected failures

Run this cell with `RUN_REGRESSIONS=True` in the supported HPC environment. It needs no real imagery/labels and can run directly after imports/configuration. A separate interpreter avoids notebook multiprocessing/import state; fixtures use temporary files, never modify your source rasters/GPKGs, and intentionally simulate missing coverage and publication failures.

The full chip suite exercises the matrix above. Polar-specific modules are `test_chip_polar`, `test_chip_seams`, and `test_chip_poles`; label planning/materialization/instance/TIFF modules cover label boundaries and encodings; acquisition/assembly/publication/creation/splits cover missing bands, failure isolation, rollback, seeding and multiprocessing. These tests do not establish every cross-product combination on real polar data.

Read both saved logs. **A dependency-skipped test is not validation**, even if unittest prints OK. Unexpected exceptions must be failures; expected rejection tests assert the appropriate contract rather than accepting any failed run. Training-loader smoke tests can take longer than geometry tests.


In [ ]:
import json
import os
import subprocess

if RUN_REGRESSIONS:
    regression_root = OUTPUT_BASE_DIR / ("regressions_" + datetime.now().strftime("%Y%m%dT%H%M%S_%f"))
    regression_root.mkdir(parents=True, exist_ok=False)
    env = dict(os.environ, PYTHONPATH=str(repo_root) + os.pathsep + os.environ.get("PYTHONPATH", ""))
    commands = {
        "chip": [sys.executable, "-m", "unittest", "discover",
                 "-s", "lfm/data_processing/tests/chip", "-t", ".", "-p", "test_chip*.py", "-v"],
        "tiling": [sys.executable, "-m", "unittest", "-v",
                   "lfm.data_processing.tests.tiling.test_grid_router",
                   "lfm.data_processing.tests.tiling.test_grid_tile_def",
                   "lfm.data_processing.tests.tiling.test_tiling_coverage"],
    }
    regression_status = {}
    for name, command in commands.items():
        log = regression_root / f"{name}.log"
        print(f"Running {name}; full output: {log}", flush=True)
        with log.open("w") as stream:
            completed = subprocess.run(command, cwd=repo_root, env=env, stdout=stream, stderr=subprocess.STDOUT)
        output = log.read_text()
        status = "FAILED" if completed.returncode else "INCOMPLETE (skips)" if "skipped=" in output else "PASSED"
        regression_status[name] = {"status": status, "returncode": completed.returncode, "log": str(log)}
        print(name, status, "\n" + "\n".join(output.splitlines()[-8:]))
    (regression_root / "summary.json").write_text(json.dumps(regression_status, indent=2) + "\n")
    if any(item["status"] != "PASSED" for item in regression_status.values()):
        raise RuntimeError(f"Validation has failures/skips; inspect {regression_root}")
else:
    print("Synthetic regression checks: NOT RUN")
